In [0]:
# path = workspace.datajoins.data_joins

In [0]:
dbutils.fs.ls("/Volumes/workspace/datajoins/data_joins")

[FileInfo(path='dbfs:/Volumes/workspace/datajoins/data_joins/JOINS SQL/', name='JOINS SQL/', size=0, modificationTime=1778029587461)]

In [0]:
%fs ls /Volumes/workspace/datajoins/data_joins

path,name,size,modificationTime
dbfs:/Volumes/workspace/datajoins/data_joins/JOINS SQL/,JOINS SQL/,0,1778029588060


In [0]:

df_files = spark.createDataFrame(
    dbutils.fs.ls
    ("/Volumes/workspace/datajoins/data_joins/")
    )
display(df_files)


path,name,size,modificationTime
dbfs:/Volumes/workspace/datajoins/data_joins/JOINS SQL/,JOINS SQL/,0,1778029589478


In [0]:
from pyspark.sql.functions import col, broadcast

In [0]:
base_path = "/Volumes/workspace/datajoins/data_joins/JOINS SQL/"

In [0]:
customers = spark.read.csv(base_path + "customers_medium.csv", header = True, inferSchema= True)

menuitems = spark.read.csv(base_path + "menu_items.csv", header = True, inferSchema= True)


orders = spark.read.csv(base_path + "order_items (2).csv", header = True, inferSchema= True)



orderMedium = spark.read.csv(base_path + "orders_medium.csv", header = True, inferSchema= True)



restaurants = spark.read.csv(base_path + "restaurants.csv", header = True, inferSchema= True)


In [0]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [0]:
# Customers who have placed orders
inner_df = customers.join(orders, "customer_id", "inner")
display(inner_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7829222276400522>, line 3
      1 # Customers who have placed orders
      2 inner_df = customers.join(orders, "customer_id", "inner")
----> 3 display(inner_df)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks support"
     95 

In [0]:
#find the all customers and their orders (inculding those who have nevered ordered)

display(left_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7829222276400523>, line 3
      1 #find the all customers and their orders (inculding those who have nevered ordered)
----> 3 display(left_df)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks support"
     95       ) from e
---

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7829222276400524>, line 1
----> 1 display(left_df)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks support"
     95       ) from e
---> 96 if df.isStreaming:
     97     self.cf_helper.display_streaming_dataframe(df, config, s

In [0]:
# find all the orders even if the customers data is missing
# show all customers and all orders even if they do not match

In [0]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)



In [0]:
orders.printSchema()
order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)



---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7829222276400527>, line 2
      1 orders.printSchema()
----> 2 order_items.printSchema()

NameError: name 'order_items' is not defined

In [0]:
#find the order that have no items
missing_items_df = orders.join (order_items, "order_id","left") \
    .filter(col("item_id").isNull())

display(missing_items_df)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7829222276400528>, line 2
      1 #find the order that have no items
----> 2 missing_items_df = orders.join (order_items, "order_id","left") \
      3     .filter(col("item_id").isNull())
      5 display(missing_items_df)

NameError: name 'order_items' is not defined

In [0]:
final_df = orders \
    .join(customers,"customer_id") \
    .join(order_items,"order_id") \
    
display(final_df)
    


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7829222276400529>, line 3
      1 final_df = orders \
      2     .join(customers,"customer_id") \
----> 3     .join(order_items,"order_id") \
      5 display(final_df)

NameError: name 'order_items' is not defined

In [0]:
# Broadcast Join - optimize  join between larger and small tables

from pyspark.sql.functions import broadcast
optimize_df = orders.join (broadcast(customers), "customer_id")

display(optimize_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7829222276400530>, line 6
      3 from pyspark.sql.functions import broadcast
      4 optimize_df = orders.join (broadcast(customers), "customer_id")
----> 6 display(optimize_df)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks